In [1]:
!pip install exifread


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


# Extracting GPS Information from Images

(You will need to modify this script based on how your dataset is stored in order to execute the code.).


In [2]:
import os

# train_or_test_or_validation could be either train, test, or validation.
train_or_test_or_validation = "validation" # it could also be test or validation
PATH_TO_YOUR_DATA_FOLDER = r"D:\UPenn\CIS-5190\project\data"
directory_path = f"{PATH_TO_YOUR_DATA_FOLDER}\{train_or_test_or_validation}"
output_csv = "metadata.csv"
output_csv = os.path.join(directory_path, output_csv)

In [2]:
import json
import exifread, csv

def get_exif_data(image_path):
    with open(image_path, 'rb') as image_file:
        tags = exifread.process_file(image_file)
    return tags

def export_exif_to_json(exif_data, output_file):
    # Convert tags to a serializable format
    exif_data_serializable = {str(tag): str(value) for tag, value in exif_data.items()}
    with open(output_file, 'w') as json_file:
        json.dump(exif_data_serializable, json_file, indent=4)

In [3]:
# Function to convert GPS coordinates in degrees, minutes, and seconds to decimal degrees
def convert_to_decimal_degrees(value):
    d, m, s = value.values
    return d.num / d.den + (m.num / m.den) / 60 + (s.num / s.den) / 3600

### You will need to create subfolders in {PATH_TO_YOUR_DATA_FOLDER} for each split (train/test/validation) or just (train/test). Next, place the corresponding images into each split after randomly shuffling them. Then, create a metadata.csv file for each split and place it in the corresponding directory. Note that the current code only works for jpeg images. If the exported images are in some other format, convert them to .jpg before running this code.

In [5]:
with open(output_csv, mode='w', newline='') as csv_file:
    fieldnames = ['file_name', 'Latitude', 'Longitude']
    writer = csv.DictWriter(csv_file, fieldnames=fieldnames)
    
    # Write the header row

    writer.writeheader()
    for filename in os.listdir(directory_path):
        print(f"Processing image file: {filename}")
        if os.path.isfile(os.path.join(directory_path, filename)):
            exif_data = get_exif_data(os.path.join(directory_path, filename))
            print(f"EXIF data found: {exif_data}")
            if exif_data:
                
                gps_latitude = exif_data.get('GPS GPSLatitude', None)
                gps_latitude_ref = exif_data.get('GPS GPSLatitudeRef', None)
                gps_longitude = exif_data.get('GPS GPSLongitude', None)
                gps_longitude_ref = exif_data.get('GPS GPSLongitudeRef', None)
                if gps_latitude and gps_longitude:
                    # Convert latitude and longitude to decimal degrees
                    latitude = convert_to_decimal_degrees(gps_latitude)
                    longitude = convert_to_decimal_degrees(gps_longitude)

                    # Adjust for N/S and E/W reference
                    if gps_latitude_ref.values[0] == 'S':
                        latitude = -latitude
                    if gps_longitude_ref.values[0] == 'W':
                        longitude = -longitude

                    # Write the data to the CSV file
                    writer.writerow({'file_name': filename, 'Latitude': latitude, 'Longitude': longitude})

File format not recognized.


Processing image file: image_0.jpg
EXIF data found: {'Image ImageWidth': (0x0100) Long=4000 @ 18, 'Image ImageLength': (0x0101) Long=3000 @ 30, 'Image Make': (0x010F) ASCII=samsung @ 158, 'Image Model': (0x0110) ASCII=SM-A315F @ 166, 'Image XResolution': (0x011A) Ratio=72 @ 176, 'Image YResolution': (0x011B) Ratio=72 @ 184, 'Image ResolutionUnit': (0x0128) Short=Pixels/Inch @ 90, 'Image Software': (0x0131) ASCII=A315FXXS5DXB3 @ 192, 'Image DateTime': (0x0132) ASCII=2024:09:23 16:06:58 @ 206, 'Image YCbCrPositioning': (0x0213) Short=Centered @ 126, 'Image ExifOffset': (0x8769) Long=226 @ 138, 'GPS GPSLatitudeRef': (0x0001) ASCII=N @ 670, 'GPS GPSLatitude': (0x0002) Ratio=[39, 57, 148347/25000] @ 738, 'GPS GPSLongitudeRef': (0x0003) ASCII=W @ 694, 'GPS GPSLongitude': (0x0004) Ratio=[75, 11, 32081999/1000000] @ 762, 'GPS GPSAltitudeRef': (0x0005) Byte=1 @ 718, 'GPS GPSAltitude': (0x0006) Ratio=7 @ 786, 'Image GPSInfo': (0x8825) Long=660 @ 150, 'EXIF ExposureTime': (0x829A) Ratio=1/60 @ 53

# Uploading and Reading a Dataset on Hugging Face

In [2]:
!pip install datasets


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


### You need to ensure that your Hugging Face token has both read and write access to your repository and Hugging Face organization.

In [3]:
!pip install huggingface_hub


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# test the CSV
import pandas as pd
train_metadata = pd.read_csv(f'{PATH_TO_YOUR_DATA_FOLDER}/train/metadata.csv')
# validation_metadata = pd.read_csv(f'{PATH_TO_YOUR_DATA_FOLDER}/validation/metadata.csv')
# test_metadata = pd.read_csv(f'{PATH_TO_YOUR_DATA_FOLDER}/test/metadata.csv')

print(train_metadata)
# print(validation_metadata)
# print(test_metadata)

In [3]:
from datasets import load_dataset
dataset = load_dataset("imagefolder", data_dir=PATH_TO_YOUR_DATA_FOLDER)
dataset.push_to_hub("cis519/dataset_v2", private=True)

Resolving data files:   0%|          | 0/501 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/101 [00:00<?, ?it/s]

ValueError: 'Content_Types' does not appear to be an IPv4 or IPv6 address

In [ ]:
### After uploading, you can access your data using this code
dataset_test = load_dataset("{YOUR ORGANIZATION NAME}/{DATASET NAME}", split="{YOUR SPLIT}")